In [0]:
import numpy as np
import os,sys
import six.moves.urllib as urllib
import sys
import tarfile
import tensorflow as tf
import zipfile

from distutils.version import StrictVersion
from collections import defaultdict
from io import StringIO
from matplotlib import pyplot as plt
from PIL import Image

if StrictVersion(tf.__version__) < StrictVersion('1.9.0'):
  raise ImportError('Please upgrade your TensorFlow installation to v1.9.* or later!')

In [0]:
# This is needed to display the images.
%matplotlib inline

In [3]:
!rm -rf *
!git clone https://github.com/tensorflow/models.git md --recursive

Cloning into 'md'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 24299 (delta 35), reused 12 (delta 5), pack-reused 24242
Receiving objects: 100% (24299/24299), 563.39 MiB | 28.55 MiB/s, done.
Resolving deltas: 100% (14383/14383), done.
Checking out files: 100% (2768/2768), done.
Submodule 'tensorflow' (https://github.com/tensorflow/tensorflow.git) registered for path 'research/syntaxnet/tensorflow'
Cloning into '/content/md/research/syntaxnet/tensorflow'...
remote: Enumerating objects: 1, done.        
remote: Counting objects: 100% (1/1), done.        
remote: Total 513689 (delta 0), reused 0 (delta 0), pack-reused 513688        
Receiving objects: 100% (513689/513689), 297.28 MiB | 22.57 MiB/s, done.
Resolving deltas: 100% (412205/412205), done.
Submodule path 'research/syntaxnet/tensorflow': checked out '8753e2ebde6c58b56675cc19ab7ff83072824a62'


In [4]:
!git clone https://github.com/cocodataset/cocoapi.git
!cd cocoapi/PythonAPI && make && cp -rv pycocotools ../../md/research/

Cloning into 'cocoapi'...
remote: Enumerating objects: 947, done.
remote: Total 947 (delta 0), reused 0 (delta 0), pack-reused 947
Receiving objects: 100% (947/947), 11.69 MiB | 12.01 MiB/s, done.
Resolving deltas: 100% (565/565), done.
python setup.py build_ext --inplace
running build_ext
cythoning pycocotools/_mask.pyx to pycocotools/_mask.c
/usr/local/lib/python3.6/dist-packages/Cython/Compiler/Main.py:367: FutureWarning: Cython directive 'language_level' not set, using 2 for now (Py2). This will change in a later release! File: /content/cocoapi/PythonAPI/pycocotools/_mask.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
building 'pycocotools._mask' extension
creating build
creating build/common
creating build/temp.linux-x86_64-3.6
creating build/temp.linux-x86_64-3.6/pycocotools
x86_64-linux-gnu-gcc -pthread -DNDEBUG -g -fwrapv -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2 -fPIC -I/usr/local/lib/python3.6/dist-packages

In [0]:
!cd md/research && protoc object_detection/protos/*.proto --python_out=.

In [6]:
!cd md/research && python setup.py install 

running install
running bdist_egg
running egg_info
creating object_detection.egg-info
writing object_detection.egg-info/PKG-INFO
writing dependency_links to object_detection.egg-info/dependency_links.txt
writing requirements to object_detection.egg-info/requires.txt
writing top-level names to object_detection.egg-info/top_level.txt
writing manifest file 'object_detection.egg-info/SOURCES.txt'
writing manifest file 'object_detection.egg-info/SOURCES.txt'
installing library code to build/bdist.linux-x86_64/egg
running install_lib
running build_py
creating build
creating build/lib
creating build/lib/object_detection
copying object_detection/model_main.py -> build/lib/object_detection
copying object_detection/inputs.py -> build/lib/object_detection
copying object_detection/inputs_test.py -> build/lib/object_detection
copying object_detection/model_tpu_main.py -> build/lib/object_detection
copying object_detection/model_hparams.py -> build/lib/object_detection
copying object_detection/eval_

In [7]:
!cd md/research/slim && python setup.py install

running install
running bdist_egg
running egg_info
creating slim.egg-info
writing slim.egg-info/PKG-INFO
writing dependency_links to slim.egg-info/dependency_links.txt
writing top-level names to slim.egg-info/top_level.txt
writing manifest file 'slim.egg-info/SOURCES.txt'
writing manifest file 'slim.egg-info/SOURCES.txt'
installing library code to build/bdist.linux-x86_64/egg
running install_lib
running build_py
creating build
creating build/lib
creating build/lib/preprocessing
copying preprocessing/lenet_preprocessing.py -> build/lib/preprocessing
copying preprocessing/preprocessing_factory.py -> build/lib/preprocessing
copying preprocessing/vgg_preprocessing.py -> build/lib/preprocessing
copying preprocessing/__init__.py -> build/lib/preprocessing
copying preprocessing/cifarnet_preprocessing.py -> build/lib/preprocessing
copying preprocessing/inception_preprocessing.py -> build/lib/preprocessing
creating build/lib/datasets
copying datasets/download_and_convert_flowers.py -> build/lib

In [8]:
!cd md/research && python object_detection/builders/model_builder_test.py

......................
----------------------------------------------------------------------
Ran 22 tests in 0.120s

OK


In [9]:
!mv md/research/object_detection ./
!mv md/research/setup.py ./
!rm -rf md
!ls

cocoapi  object_detection  setup.py


In [11]:
!python object_detection/builders/model_builder_test.py

......................
----------------------------------------------------------------------
Ran 22 tests in 0.121s

OK


In [0]:
sys.path.append("object_detection")

In [0]:
# This is needed since the notebook is stored in the object_detection folder.
sys.path.append("..")
from object_detection.utils import ops as utils_ops
from object_detection import utils
from utils import label_map_util

from utils import visualization_utils as vis_util

In [0]:
!pip install -q opencv-python
import cv2,time
print('Using OpenCV version',cv2.__version__)

Using OpenCV version 3.4.3


In [0]:
!wget -nc https://github.com/manuhg/masknet/raw/master/input_video_vs.mp4
input_file='input_video_vs.mp4'

File ‘input_video_vs.mp4’ already there; not retrieving.



In [0]:
def extract_frames(input_file,class_labels):
  cap = cv2.VideoCapture(input_file)
  if (cap.isOpened()== False): 
    print("Error opening video file")
  i,count=0,0
  fps = video.get(cv2.CAP_PROP_FPS)
  t1 = time.time()
  while(cap.isOpened()):
    ret, frame = cap.read()
    i+=1
    if ret == True:
      frame,has_class_labels = predictor.get_predictions(frame,class_labels)
      if has_class_labels:
        cv2.imwrite(input_file+'-'+str(int(i/fps))+':'+str(i%fps)+'.jpg',frame)
      #cv2.imshow('Frame',frame)
      #plt.imshow(frame)
      
    else:
      break
  t2 = time.time()
  print('Processing speed:',(t2-t1)/i)
  cap.release()
  #cv2.destroyAllWindows()